In [1]:
import random
import torch
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader

from mingpt.utils import set_seed
from mingpt.model import GPT
from mingpt.trainer import Trainer

set_seed(3407)

In [2]:
DIGITS = 10
SEP = 10  # separator między segmentami
VOCAB_SIZE = 11  # 0..9 + SEP

In [3]:
def int_to_fixed_digits(x: int, width: int):
    s = str(x)
    s = (width - len(s)) * "0" + s
    return [int(ch) for ch in s]

def digits3_to_int(d3):
    return int("".join(map(str, d3)))

In [4]:
def random_mul_long_instance():
    """
    Zwraca sekwencję tokenów:
    A(3) B(3) SEP P0(6) SEP P1(6) SEP P2(6) SEP SUM(6)
    gdzie:
    - B = [d2 d1 d0] (setki, dziesiątki, jedności)
    - P0 = A * d0 (uwzględnia shift 0) => 6 cyfr padded
    - P1 = A * d1 * 10 (shift 1)       => 6 cyfr padded
    - P2 = A * d2 * 100 (shift 2)      => 6 cyfr padded
    - SUM = A*B                        => 6 cyfr padded
    """
    A = [random.randint(0, 9) for _ in range(3)]
    B = [random.randint(0, 9) for _ in range(3)]

    a = digits3_to_int(A)
    b = digits3_to_int(B)

    d2, d1, d0 = B[0], B[1], B[2]  # setki, dziesiątki, jedności

    p0 = a * d0
    p1 = a * d1 * 10
    p2 = a * d2 * 100
    s  = a * b

    P0 = int_to_fixed_digits(p0, 6)
    P1 = int_to_fixed_digits(p1, 6)
    P2 = int_to_fixed_digits(p2, 6)
    S  = int_to_fixed_digits(s,  6)

    seq = A + B + [SEP] + P0 + [SEP] + P1 + [SEP] + P2 + [SEP] + S
    return seq


In [5]:

class MulLongDataset(Dataset):
    """
    Uczymy next-token prediction na całej sekwencji,
    ale loss liczymy tylko na części wynikowej:
    od pierwszego tokena po SEP (czyli zaczynamy przewidywać P0).
    """

    def __init__(self, split: str):
        assert split in {"train", "test"}
        self.split = split

        # stałe długości:
        # A(3) + B(3) + SEP(1) + 4 segmenty po 6 cyfr + 3 SEP-y między nimi?
        # Dokładnie: A3 B3 SEP + P0(6) SEP + P1(6) SEP + P2(6) SEP + S(6)
        # = 3+3+1 + 6+1 + 6+1 + 6+1 + 6 = 34 tokeny
        self.total_len = 34
        self.prompt_len = 7  # A(3)+B(3)+SEP

    def __len__(self):
        return 10000

    def get_vocab_size(self):
        return VOCAB_SIZE

    def get_block_size(self):
        # x = cat[:-1] => total_len - 1
        return self.total_len - 1  # 33

    def __getitem__(self, idx):
        while True:
            seq = random_mul_long_instance()  # 34 tokeny
            # split po hashu samego "promptu" (A+B)
            h = hash(str(seq[:6]))
            inp_split = "test" if h % 4 == 0 else "train"
            if inp_split == self.split:
                break

        x = torch.tensor(seq[:-1], dtype=torch.long)  # 33
        y = torch.tensor(seq[1:],  dtype=torch.long)  # 33

        # maskujemy loss na prompt (A+B+SEP), ale pamiętaj:
        # y jest przesunięte o 1, więc trzeba zignorować pierwsze (prompt_len-1) pozycji y
        y[: self.prompt_len - 1] = -1
        return x, y
    

In [6]:
train_dataset = MulLongDataset("train")
test_dataset  = MulLongDataset("test")

model_config = GPT.get_default_config()
model_config.model_type = "gpt-micro"  # często wystarcza; jak trzeba => gpt-nano
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()

model = GPT(model_config)
print("block_size =", model_config.block_size, "vocab_size =", model_config.vocab_size)


number of parameters: 0.80M
block_size = 33 vocab_size = 11


In [7]:

train_config = Trainer.get_default_config()
train_config.learning_rate = 3e-4
train_config.max_iters = 5000
train_config.num_workers = 0

trainer = Trainer(train_config, model, train_dataset)


running on device cpu


In [8]:

def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")

trainer.set_callback("on_batch_end", batch_end_callback)
trainer.run()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


iter_dt 0.00ms; iter 0: train loss 2.42585
iter_dt 164.54ms; iter 100: train loss 1.23618
iter_dt 219.84ms; iter 200: train loss 1.02842
iter_dt 165.63ms; iter 300: train loss 0.92198
iter_dt 169.31ms; iter 400: train loss 0.80354
iter_dt 187.37ms; iter 500: train loss 0.75746
iter_dt 172.09ms; iter 600: train loss 0.64118
iter_dt 167.52ms; iter 700: train loss 0.50558
iter_dt 166.29ms; iter 800: train loss 0.46985
iter_dt 154.17ms; iter 900: train loss 0.45775
iter_dt 153.69ms; iter 1000: train loss 0.36636
iter_dt 147.18ms; iter 1100: train loss 0.35793
iter_dt 158.44ms; iter 1200: train loss 0.29769
iter_dt 164.05ms; iter 1300: train loss 0.28220
iter_dt 154.16ms; iter 1400: train loss 0.26153
iter_dt 160.09ms; iter 1500: train loss 0.24156
iter_dt 161.07ms; iter 1600: train loss 0.19412
iter_dt 145.71ms; iter 1700: train loss 0.19728
iter_dt 165.56ms; iter 1800: train loss 0.16776
iter_dt 183.19ms; iter 1900: train loss 0.15968
iter_dt 153.12ms; iter 2000: train loss 0.14456
iter_d

In [9]:
# now let's perform some evaluation
model.eval()
None

In [10]:

def eval_mul_long_split(trainer, split, max_batches=50):
    dataset = {"train": train_dataset, "test": test_dataset}[split]
    results = []

    loader = DataLoader(dataset, batch_size=100, num_workers=0, drop_last=False)

    model.eval()
    for b, (x, y) in enumerate(loader):
        if b >= max_batches:
            break

        x = x.to(trainer.device)
        y = y.to(trainer.device)

        # prompt to A(3) B(3) SEP => 7 tokenów
        inp = x[:, : dataset.prompt_len]

        # co jest "prawdziwą odpowiedzią"?
        # Chcemy, żeby model wygenerował resztę sekwencji:
        # total_len - prompt_len tokenów
        target_len = dataset.total_len - dataset.prompt_len

        # prawdziwy "tail" sekwencji jest w y, ale najprościej porównać
        # z tym, co powinno się pojawić po prompt:
        # z x/y dostaniesz to tak:
        # pełna sekwencja = [inp] + [tail]
        # tail zaczyna się w x od indeksu prompt_len (bo x ma cat[:-1])
        true_tail = x[:, dataset.prompt_len:]  # długość = (total_len-1) - prompt_len = target_len-1
        # brakuje jeszcze ostatniego tokena tail, więc do porównania weźmiemy z y:
        # ostatni token całej sekwencji jest w y jako ostatni element
        last_token = y[:, -1:]                # 1 token
        true_tail_full = torch.cat([true_tail, last_token], dim=1)  # target_len

        # generowanie
        cat = model.generate(inp, target_len, do_sample=False)
        gen_tail = cat[:, -target_len:]

        correct = (gen_tail == true_tail_full).all(1).cpu()
        results.extend(correct.int().tolist())

    rt = torch.tensor(results, dtype=torch.float)
    print("%s final score: %d/%d = %.2f%% correct" % (split, rt.sum(), len(results), 100 * rt.mean()))
    return rt.sum()

with torch.no_grad():
    train_score = eval_mul_long_split(trainer, "train", max_batches=50)
    test_score  = eval_mul_long_split(trainer, "test",  max_batches=50)

train final score: 4387/5000 = 87.74% correct
test final score: 4349/5000 = 86.98% correct
